In [1]:
import pandas as pd
import numpy as np

In [2]:
orders_detail = pd.read_excel("Company ShopX - Order Report.xlsx")
product_weights = pd.read_excel("Company ShopX - Product Weight.xlsx")
invoice = pd.read_excel("Courier Company - Invoice.xlsx")
zone = pd.read_excel("Company ShopX - Warehouse&Customer Pin Code and Zone details.xlsx")
rate = pd.read_excel("Courier Company - Rates.xlsx")

In [3]:
zone = zone.drop_duplicates(subset=["Store House Pincode","Customer Area Code"])

In [4]:
df = pd.merge(orders_detail,product_weights,on="Product Code")

In [5]:
df["Line Weight (kg)"] = df["Units Ordered"] * df["Product Weight (g)"] / 1000

In [6]:
df["Order Weight"] = np.ceil(df["Line Weight (kg)"] / 0.5) * 0.5

In [7]:
orders_invo = pd.merge(df,invoice,on="Order ID")

In [8]:
orders_invoice = pd.merge(orders_invo,zone,on=["Store House Pincode","Customer Area Code"],suffixes=("_Courier","_ShopX"))

In [9]:
def Amount_Cal(x):
    if pd.isna(x['Order Weight']) or x['Order Weight'] <= 0 or pd.isna(x["Delivery Zone_ShopX"]):
        return np.nan
    else:
        slab = np.ceil(x['Order Weight']/0.5)
        if slab <= 0:
            return np.nan
        else:
            zone = x['Delivery Zone_ShopX']
            amount = rate[f'fwd_{zone}_fixed']
            if slab > 1:
                additional_slab = slab - 1
                amount = amount + rate[f'fwd_{zone}_additional'] * additional_slab
            if "RTO" in x["Freight Type"]:
                additional_slab = slab - 1
                amount = amount + rate[f"rto_{zone}_fixed"] * additional_slab
                if slab > 1:
                    amount = amount + rate[f"rto_{zone}_additional"]
    return amount

In [10]:
orders_invoice["Total Amount (Rs.)_Shopx"]= orders_invoice.apply(Amount_Cal,axis=1)

In [11]:
calculation = pd.DataFrame({
    "Order ID" : orders_invoice["Order ID"],
    "AWB Number" : orders_invoice["AWB Code (Airway Bill Number) "],
    "Total weight as per ShopX (KG)" : round(orders_invoice["Line Weight (kg)"],2),
    "Weight slab as per ShopX (KG)" : orders_invoice["Order Weight"],
    "Total weight as per Courier Company (KG)" : orders_invoice["Chargeable Weight"],
    "Weight slab charged by Courier Company (KG)" : np.ceil(orders_invoice["Chargeable Weight"] / 0.5) * 0.5,
    "Delivery Zone as per ShopX" : orders_invoice["Delivery Zone_ShopX"],
    "Delivery Zone charged by Courier Company" : orders_invoice["Delivery Zone_Courier"],
    "Expected Charge as per ShopX (Rs.)" : orders_invoice["Total Amount (Rs.)_Shopx"],
    "Charges Billed by Courier Company (Rs.)": orders_invoice["Total Amount (Rs.)"],
    "Difference Between Expected Charges and Billed Charges (Rs.)" : orders_invoice["Total Amount (Rs.)"] - orders_invoice["Total Amount (Rs.)_Shopx"]
}
)

In [12]:
def charge_status(diff):

    if pd.isna(diff):
        return "Needs Review"

    if abs(diff) <= 0.01:
        return "Correctly Charged"

    elif diff > 0:
        return "Overcharged"

    else:
        return "Undercharged"

In [13]:
calculation["Charged Status"] = calculation["Difference Between Expected Charges and Billed Charges (Rs.)"].apply(charge_status)

In [14]:
calculation.head(1)

,Order ID,AWB Number,Total weight as per ShopX (KG),Weight slab as per ShopX (KG),Total weight as per Courier Company (KG),Weight slab charged by Courier Company (KG),Delivery Zone as per ShopX,Delivery Zone charged by Courier Company,Expected Charge as per ShopX (Rs.),Charges Billed by Courier Company (Rs.),Difference Between Expected Charges and Billed Charges (Rs.),Charged Status
0,2001827036,1091122418320,0.13,0.5,1.6,2.0,e,b,56.6,117.9,61.3,Overcharged


In [23]:
summary = pd.pivot_table(calculation,index="Charged Status",values=["Order ID","Difference Between Expected Charges and Billed Charges (Rs.)"],
              aggfunc={"Order ID": "count","Difference Between Expected Charges and Billed Charges (Rs.)": "sum"})
summary = summary.rename(columns={
    "Order ID": "Count",
    "Difference Between Expected Charges and Billed Charges (Rs.)": "Amount (Rs.)"
})
summary["Amount (Rs.)"] = summary["Amount (Rs.)"].abs()
summary.reset_index(inplace=True)

In [24]:
sum_correct = round(calculation[calculation['Charged Status'] == "Correctly Charged"]["Charges Billed by Courier Company (Rs.)"].sum(),1)

In [25]:
summary.loc[summary["Charged Status"] == "Correctly Charged","Amount (Rs.)"] = sum_correct

In [26]:
summary

,Charged Status,Amount (Rs.),Count
0,Correctly Charged,507.6,12
1,Overcharged,33330.7,382
2,Undercharged,165.2,7


In [27]:
with pd.ExcelWriter("ShopX_Solution.xlsx",engine="openpyxl") as writer:
    calculation.to_excel(writer,sheet_name="Calculation",index=False)
    summary.to_excel(writer,sheet_name="Summary",index=False)
print("Excel File Saved Successfully")

Excel File Saved Successfully
